In [8]:
%%writefile conv_kernels.cu
// convolution kernels matching the implementation of F.conv2d() with padding = 0
#include <torch/extension.h> // for interface with pytorch

#define MAXK 11

__constant__ float d_kernel[MAXK*MAXK];

// kernel 1. naive 2d convolution
__global__ void conv2d_naive_kernel(
    const float* __restrict__ input,
    const float* __restrict__ kernel,
    float* __restrict__ output,
    int H,
    int W,
    int kH,
    int kW,
    int oH,
    int oW
)
{
  int row = blockIdx.y * blockDim.y + threadIdx.y;
  int col = blockIdx.x * blockDim.x + threadIdx.x;

  if (row >= oH || col >= oW) return;
  float Pacc = 0.0f;
  for (int kh=0;kh < kH; ++kh)
  {
    for (int kw=0;kw < kW; ++kw)
    {
      Pacc += input[(row+kh) * W + col+kw] * kernel[kh*kW + kw];
    }
  }
  output[row * oW + col] = Pacc;
}

// kernel 2 : caching the kernels in constant cache
__global__ void conv2d_constant_kernel(
    const float* __restrict__ input,
    float* __restrict__ output,
    int H,
    int W,
    int kH,
    int kW,
    int oH,
    int oW
)
{
  int row = blockIdx.y * blockDim.y + threadIdx.y;
  int col = blockIdx.x * blockDim.x + threadIdx.x;

  if (row >= oH || col >= oW) return;
  float Pacc = 0.0f;
  for (int kh=0;kh < kH; ++kh)
  {
    for (int kw=0;kw < kW; ++kw)
    {
      Pacc += input[(row+kh) * W + col+kw] * d_kernel[kh*kW + kw];
    }
  }
  output[row * oW + col] = Pacc;
}

// kernel 3 : tiled convolution
// we design the approach where the TILE_DIM is number of output cells written to per tile and the
// number of input cells is TILE_DIM + K - 1
template <int TILE_DIM_Y, int TILE_DIM_X>
__global__ void conv2d_tiled_kernel(
    const float* __restrict__ input,
    float* __restrict__ output,
    int H,
    int W,
    int kH,
    int kW,
    int oH,
    int oW
)
{
  constexpr int IN_TILE_X = TILE_DIM_X + MAXK - 1;
  constexpr int IN_TILE_Y = TILE_DIM_Y + MAXK - 1;
  __shared__ float s_tile[IN_TILE_Y][IN_TILE_X];
  int tx = threadIdx.x;
  int ty = threadIdx.y;
  int row = blockIdx.y * TILE_DIM_Y + ty;
  int col = blockIdx.x * TILE_DIM_X + tx;

  for (int i=ty; i < IN_TILE_Y; i+=TILE_DIM_Y)
  {
    for (int j=tx; j < IN_TILE_X; j+=TILE_DIM_X)
    {
      int in_row = blockIdx.y * TILE_DIM_Y + i;
      int in_col = blockIdx.x * TILE_DIM_X + j;
      if (in_row < H && in_col < W)
      {
        s_tile[i][j] = input[in_row * W + in_col];
      }
      else
      {
        s_tile[i][j] = 0.0f;
      }
    }
  }
  __syncthreads();

  if (row < oH && col < oW)
  {
    float Pacc = 0.0f;
    for (int kh = 0; kh < kH; ++kh)
    {
      for (int kw = 0; kw < kW; ++kw)
      {
        Pacc += s_tile[ty+kh][tx+kw] * d_kernel[kh * kW + kw];
      }
    }
    output[row * oW + col] = Pacc;
  }
}

//wrappers
// the python calls these wrappers to run the kernel
torch::Tensor launch_naive(
  torch::Tensor input, // 2D HXW float32 contiguous
  torch::Tensor kernel // 2D kHXkW float32 contiguous
)
{
  int H = input.size(0);
  int W = input.size(1);
  int kH = kernel.size(0);
  int kW = kernel.size(1);
  int oH = H - kH + 1;
  int oW = W - kW + 1;
  torch::Tensor output = torch::zeros({oH, oW}, input.options());

  int threadX = 16;
  int threadY = 16;
  int blockX = (oW + threadX - 1) / threadX;
  int blockY = (oH + threadY - 1) / threadY;

  dim3 block(threadX, threadY);
  dim3 grid(blockX, blockY);
  conv2d_naive_kernel<<<grid, block>>>(
    input.data_ptr<float>(),
    kernel.data_ptr<float>(),
    output.data_ptr<float>(),
    H,
    W,
    kH,
    kW,
    oH,
    oW
  );
  return output;
}
// function to copy tensor from device to constant memory of the device
void copyKernelToConstant(torch::Tensor kernel)
{
  cudaMemcpyToSymbol(
    d_kernel, // global constant memory
    kernel.data_ptr<float>(),
    kernel.numel() * sizeof(float),
    0,
    cudaMemcpyDeviceToDevice
  );
}
torch::Tensor launch_constantKernel(
  torch::Tensor input, // 2D HXW float32 contiguous
  int kH,
  int kW
)
{
  int H = input.size(0);
  int W = input.size(1);
  int oH = H - kH + 1;
  int oW = W - kW + 1;
  torch::Tensor output = torch::zeros({oH, oW}, input.options());

  int threadX = 16;
  int threadY = 16;
  int blockX = (oW + threadX - 1) / threadX;
  int blockY = (oH + threadY - 1) / threadY;

  dim3 block(threadX, threadY);
  dim3 grid(blockX, blockY);


  conv2d_constant_kernel<<<grid, block>>>(
    input.data_ptr<float>(),
    output.data_ptr<float>(),
    H,
    W,
    kH,
    kW,
    oH,
    oW
  );
  return output;
}

torch::Tensor launch_tiledKernel16(
  torch::Tensor input, // 2D HXW float32 contiguous
  int kH,
  int kW
)
{
  int H = input.size(0);
  int W = input.size(1);
  int oH = H - kH + 1;
  int oW = W - kW + 1;
  torch::Tensor output = torch::zeros({oH, oW}, input.options());

  int threadX = 16;
  int threadY = 16;
  int blockX = (oW + threadX - 1) / threadX;
  int blockY = (oH + threadY - 1) / threadY;

  dim3 block(threadX, threadY);
  dim3 grid(blockX, blockY);


  conv2d_tiled_kernel<16, 16><<<grid, block>>>(
    input.data_ptr<float>(),
    output.data_ptr<float>(),
    H,
    W,
    kH,
    kW,
    oH,
    oW
  );
  return output;
}

torch::Tensor launch_tiledKernel32(
  torch::Tensor input, // 2D HXW float32 contiguous
  int kH,
  int kW
)
{
  int H = input.size(0);
  int W = input.size(1);
  int oH = H - kH + 1;
  int oW = W - kW + 1;
  torch::Tensor output = torch::zeros({oH, oW}, input.options());

  int threadX = 32;
  int threadY = 32;
  int blockX = (oW + threadX - 1) / threadX;
  int blockY = (oH + threadY - 1) / threadY;

  dim3 block(threadX, threadY);
  dim3 grid(blockX, blockY);


  conv2d_tiled_kernel<32, 32><<<grid, block>>>(
    input.data_ptr<float>(),
    output.data_ptr<float>(),
    H,
    W,
    kH,
    kW,
    oH,
    oW
  );
  return output;
}



PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
  m.def("launch_naive", &launch_naive, "naive 2d convolution (global memory)");
  m.def("copyKernelToConstant", &copyKernelToConstant, "copy kernel tensor from global memory to cached constant memory");
  m.def("launch_constantKernel", &launch_constantKernel, "2d convolution with kernel in constant meory");
  m.def("launch_tiledKernel16", &launch_tiledKernel16, "2d convolution with tiled kernel of tile size 16");
  m.def("launch_tiledKernel32", &launch_tiledKernel32, "2d convolution with tiled kernel of tile size 32");
}


Writing conv_kernels.cu


In [9]:
!pip install ninja

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 6.9 MB/s eta 0:00:00


In [10]:
import os
os.environ['CUDA_HOME'] = '/usr/local/cuda'
import torch
from torch.utils.cpp_extension import load

os.environ['TORCH_SHOW_CPP_EXTENSION_DEBUG'] = '1'

# Explicitly define extra compilation flags
extra_cuda_cflags = ['-gencode=arch=compute_75,code=sm_75'] # Common for Colab GPUs
extra_ldflags = ['-lcudart'] # Explicitly link CUDA runtime library, sometimes needed

# Define a custom build directory for easier inspection
build_dir = os.path.join(os.getcwd(), 'cuda_build')
os.makedirs(build_dir, exist_ok=True) # Ensure the directory exists

conv_module = load(
      name     = "conv_kernels",
      sources  = ["/content/conv_kernels.cu"],
      extra_cuda_cflags = extra_cuda_cflags,
      extra_ldflags = extra_ldflags,
      build_directory = build_dir, # Use the custom build directory
      verbose  = True
  )


Check if the kernel is mathematically accurate by comparing against F.conv2D before benchmarking its performance.

In [11]:
import torch, torch.nn.functional as F
H, W = 64, 64
K = 3
inp = torch.randn(H, W, device='cuda')
kernel = torch.randn(K, K, device = 'cuda')
print("-------TEST NAIVE KERNEL---------")
out_naive = conv_module.launch_naive(inp, kernel)
out_torch = F.conv2d(inp.unsqueeze(0).unsqueeze(0), kernel.unsqueeze(0).unsqueeze(0), padding = 0).squeeze()
max_err = (out_naive - out_torch).abs().max()
print(f"max_err = {max_err:.2e}")
print("PASS" if torch.allclose(out_naive, out_torch, atol=1e-4) else "FAIL")
print("------TEST CONST MEMORY KERNEL ---------")
conv_module.copyKernelToConstant(kernel)
out_constant = conv_module.launch_constantKernel(inp, K, K)
max_err = (out_constant - out_torch).abs().max()
print(f"max_err = {max_err:.2e}")
print("PASS" if torch.allclose(out_constant, out_torch, atol=1e-4) else "FAIL")
print("-------TEST TILED CONV (TILE SIZE 16) ------")
out_tiled16 = conv_module.launch_tiledKernel16(inp, K, K)
max_err = (out_tiled16 - out_torch).abs().max()
print(f"max_err = {max_err:.2e}")
print("-------TEST TILED CONV (TILE SIZE 32) ------")
out_tiled32 = conv_module.launch_tiledKernel32(inp, K, K)
max_err = (out_tiled32 - out_torch).abs().max()
print(f"max_err = {max_err:.2e}")


-------TEST NAIVE KERNEL---------
max_err = 0.00e+00
PASS
------TEST CONST MEMORY KERNEL ---------
max_err = 0.00e+00
PASS
-------TEST TILED CONV (TILE SIZE 16) ------
max_err = 0.00e+00
-------TEST TILED CONV (TILE SIZE 32) ------
max_err = 0.00e+00


Benchmarking the kernels for the different configurations

In [12]:
import torch
import torch.nn.functional as F
import pandas as pd
T4_PEAK_BW_GBs = 320.0     # T4 memory bandwidth peak
T4_PEAK_GFLOPS = 8141.0    # T4 FP32 compute peak
T4_RIDGE_POINT = T4_PEAK_GFLOPS / T4_PEAK_BW_GBs   # ~25.4 FLOP/byte


In [13]:
def gpu_timer(fn, *args, warmup=3, runs=20):
      """
      Time a GPU function with CUDA events.
      warmup runs are discarded (primes L2, triggers clock boost).
      Returns average milliseconds over `runs` iterations.
      """
      for _ in range(warmup):
          fn(*args)
      torch.cuda.synchronize()          # ensure warmup fully complete

      start = torch.cuda.Event(enable_timing=True)
      end   = torch.cuda.Event(enable_timing=True)
      start.record()
      for _ in range(runs):
          fn(*args)
      end.record()
      torch.cuda.synchronize()
      return start.elapsed_time(end) / runs   # ms per run

In [14]:
def compute_metrics(N, K, time_ms):
      """
      Compute perf metrics for square NxN image, KxK kernel, valid conv (padding=0).

      Arithmetic intensity = FLOPs / minimum_bytes_moved.
      'Minimum bytes' counts each array element once — this is the ideal
      lower bound on data movement. Naive kernel moves more (redundant fetches)
      but we report it against the same denominator so all kernels are on equal footing.
      Higher effective bandwidth = less wasted DRAM traffic.
      """
      oN          = N - K + 1                          # output side length
      flops       = 2 * K * K * oN * oN               # 2 ops per MAC
      bytes_total = (N*N + K*K + oN*oN) * 4           # float32, each element once
      time_s      = time_ms / 1e3

      bw_GBs      = bytes_total / time_s / 1e9
      gflops      = flops       / time_s / 1e9
      arith_int   = flops / bytes_total                # FLOP/byte — same for all kernels
      pct_peak_bw = bw_GBs  / T4_PEAK_BW_GBs  * 100
      pct_peak_fp = gflops  / T4_PEAK_GFLOPS   * 100

      return {
          "time_ms"     : round(time_ms,   4),
          "bw_GBs"      : round(bw_GBs,    2),
          "gflops"      : round(gflops,     4),
          "arith_int"   : round(arith_int,  4),
          "pct_peak_bw" : round(pct_peak_bw, 2),
          "pct_peak_fp" : round(pct_peak_fp, 4),
      }

In [15]:
def run_naive(inp, ker):
      return conv_module.launch_naive(inp, ker)

def run_constmem(inp, ker):
      # copyKernelToConstant is called once in the sweep loop BEFORE timing starts
      return conv_module.launch_constantKernel(inp, ker.size(0), ker.size(1))

def run_tiled16(inp, ker):
      return conv_module.launch_tiledKernel16(inp, ker.size(0), ker.size(1))

def run_tiled32(inp, ker):
      return conv_module.launch_tiledKernel32(inp, ker.size(0), ker.size(1))

def run_pytorch(inp, ker):
      return F.conv2d(
          inp.unsqueeze(0).unsqueeze(0),   # [1, 1, H, W]
          ker.unsqueeze(0).unsqueeze(0),   # [1, 1, kH, kW]
          padding=0
      ).squeeze()

In [16]:
KERNELS = {
      "1_naive"      : run_naive,
      "2_constmem"   : run_constmem,
      "3_tiled16"    : run_tiled16,
      "4_tiled32"    : run_tiled32,
      "5_pytorch_ref": run_pytorch,
  }

In [17]:
KERNEL_SETUP = {
      "2_constmem": lambda inp, ker: conv_module.copyKernelToConstant(ker.contiguous()),
  }

In [18]:
N_VALUES = [512, 1024, 2048, 4096]
K_VALUES = [3, 5, 7]
WARMUP   = 3
RUNS     = 20

results = []

for N in N_VALUES:
      for K in K_VALUES:
          inp = torch.randn(N, N, device='cuda', dtype=torch.float32).contiguous()
          ker = torch.randn(K, K, device='cuda', dtype=torch.float32).contiguous()

          print(f"N={N:5d}  K={K}", end="")

          for name, fn in KERNELS.items():
              # One-time setup (e.g. constant memory copy) — not timed
              if name in KERNEL_SETUP:
                  KERNEL_SETUP[name](inp, ker)
                  torch.cuda.synchronize()   # ensure copy done before timer starts

              time_ms = gpu_timer(fn, inp, ker, warmup=WARMUP, runs=RUNS)
              metrics = compute_metrics(N, K, time_ms)

              results.append({"kernel": name, "N": N, "K": K, **metrics})
              print(f"  |  {name}: {time_ms:.3f}ms", end="")

          print()

df = pd.DataFrame(results)
print(f"\nDone. {len(df)} cases measured.")

N=  512  K=3  |  1_naive: 0.041ms  |  2_constmem: 0.035ms  |  3_tiled16: 0.042ms  |  4_tiled32: 0.042ms  |  5_pytorch_ref: 0.053ms
N=  512  K=5  |  1_naive: 0.076ms  |  2_constmem: 0.054ms  |  3_tiled16: 0.062ms  |  4_tiled32: 0.060ms  |  5_pytorch_ref: 0.069ms
N=  512  K=7  |  1_naive: 0.125ms  |  2_constmem: 0.090ms  |  3_tiled16: 0.138ms  |  4_tiled32: 0.078ms  |  5_pytorch_ref: 0.113ms
N= 1024  K=3  |  1_naive: 0.141ms  |  2_constmem: 0.130ms  |  3_tiled16: 0.159ms  |  4_tiled32: 0.158ms  |  5_pytorch_ref: 0.135ms
N= 1024  K=5  |  1_naive: 0.260ms  |  2_constmem: 0.198ms  |  3_tiled16: 0.233ms  |  4_tiled32: 0.227ms  |  5_pytorch_ref: 0.251ms
N= 1024  K=7  |  1_naive: 0.470ms  |  2_constmem: 0.330ms  |  3_tiled16: 0.356ms  |  4_tiled32: 0.290ms  |  5_pytorch_ref: 0.414ms
N= 2048  K=3  |  1_naive: 0.535ms  |  2_constmem: 0.498ms  |  3_tiled16: 0.620ms  |  4_tiled32: 0.613ms  |  5_pytorch_ref: 0.472ms
N= 2048  K=5  |  1_naive: 1.015ms  |  2_constmem: 0.769ms  |  3_tiled16: 0.898ms  |

In [19]:
SEP = "=" * 68

  # ── Execution time ──────────────────────────────────────────────────────
for K in K_VALUES:
      print(f"\n{SEP}")
      print(f"  Execution Time (ms)  —  K = {K}x{K}")
      print(SEP)
      print(df[df.K==K].pivot(index='kernel', columns='N', values='time_ms').to_string())

  # ── Effective bandwidth ─────────────────────────────────────────────────
for K in K_VALUES:
      print(f"\n{SEP}")
      print(f"  Effective Bandwidth (GB/s)  —  K = {K}x{K}  [T4 peak = 320 GB/s]")
      print(SEP)
      print(df[df.K==K].pivot(index='kernel', columns='N', values='bw_GBs').to_string())

  # ── % of T4 peak bandwidth ──────────────────────────────────────────────
for K in K_VALUES:
      print(f"\n{SEP}")
      print(f"  % of T4 Peak Bandwidth  —  K = {K}x{K}")
      print(SEP)
      print(df[df.K==K].pivot(index='kernel', columns='N', values='pct_peak_bw').to_string())

  # ── Speedup vs naive (N=1024) ───────────────────────────────────────────
print(f"\n{SEP}")
print(f"  Speedup vs Naive  —  N = 1024")
print(SEP)
sub = df[df.N == 1024].copy()
naive_t = sub[sub.kernel == '1_naive'].set_index('K')['time_ms']
sub['speedup'] = sub.apply(lambda r: round(naive_t[r.K] / r.time_ms, 2), axis=1)
print(sub.pivot(index='kernel', columns='K', values='speedup').to_string())

  # ── Time as % of PyTorch reference ─────────────────────────────────────
print(f"\n{SEP}")
print(f"  Time as % of PyTorch/cuDNN reference  —  N = 1024")
print(f"  (100% = matched cuDNN, >100% = slower than cuDNN)")
print(SEP)
ref_t = sub[sub.kernel == '5_pytorch_ref'].set_index('K')['time_ms']
sub['pct_of_ref'] = sub.apply(lambda r: round(r.time_ms / ref_t[r.K] * 100, 1), axis=1)
print(sub.pivot(index='kernel', columns='K', values='pct_of_ref').to_string())

  # ── Arithmetic intensity reminder (constant per K) ──────────────────────
print(f"\n{SEP}")
print(f"  Arithmetic Intensity (FLOP/byte)  —  same for all kernels at given K")
print(f"  T4 ridge point = {T4_RIDGE_POINT:.1f} FLOP/byte")
print(SEP)
for K in K_VALUES:
      ai = df[df.K == K]['arith_int'].iloc[0]
      region = "memory-bound" if ai < T4_RIDGE_POINT else "compute-bound"
      print(f"  K={K}:  {ai:.3f} FLOP/byte  →  {region}")


  Execution Time (ms)  —  K = 3x3
N                512     1024    2048    4096
kernel                                       
1_naive        0.0415  0.1410  0.5353  1.5057
2_constmem     0.0353  0.1296  0.4978  1.3796
3_tiled16      0.0425  0.1585  0.6197  1.6052
4_tiled32      0.0420  0.1577  0.6134  1.5893
5_pytorch_ref  0.0529  0.1346  0.4724  0.9663

  Execution Time (ms)  —  K = 5x5
N                512     1024    2048    4096
kernel                                       
1_naive        0.0763  0.2600  1.0149  2.2793
2_constmem     0.0539  0.1980  0.7685  2.0220
3_tiled16      0.0625  0.2334  0.8981  2.2960
4_tiled32      0.0596  0.2269  0.8945  2.1808
5_pytorch_ref  0.0687  0.2512  0.5645  2.1413

  Execution Time (ms)  —  K = 7x7
N                512     1024    2048    4096
kernel                                       
1_naive        0.1251  0.4699  1.0609  4.1654
2_constmem     0.0900  0.3297  0.7376  2.9847
3_tiled16      0.1380  0.3563  0.7920  3.2513
4_tiled32      0.0778